# Figures

Generates every figure and equation render cited in the thesis. All outputs are
written as `.jpg` files to the `figures/` subdirectory next to this notebook.

**Inputs**

  - `../sp500_it_processed_full.csv.gz` — long-form panel of 14 features × 47 stocks (only
    `TICKER`, `date`, `PRC`, `RSI`, `log_vol` columns are used here, and only the
    `date` column for the equity-curve x-axis).
  - `results/pipeline_a_test_daily_returns.csv` — daily returns for all five
    strategies on the Pipeline A test window (2022–2024), used by Figure 9.
  - `results/pipeline_b_test_results.csv` — per-window Sharpe for all five
    strategies across the six Pipeline B walk-forward windows, used by Figure 10.

**Outputs (written to `figures/`)**

  - `figure1_pipeline.jpg`               — overall methodological pipeline
  - `figure2_zscore_prc.jpg`             — AAPL closing price, raw vs z-score
  - `figure3_zscore_rsi.jpg`             — AAPL RSI, raw vs z-score
  - `figure4_zscore_logvol.jpg`          — AAPL log volume, raw vs z-score
  - `figure5_ppo_architecture.jpg`       — PPO actor network architecture
  - `figure6_dqn_architecture.jpg`       — DQN Q-network architecture
  - `figure7_pipeline_a.jpg`             — Pipeline A fixed train/val/test split
  - `figure8_pipeline_b.jpg`             — Pipeline B six walk-forward windows
  - `figure9_test_a_equity.jpg`          — Pipeline A test-set equity curves
  - `figure10_test_b_sharpe_bar.jpg`     — Pipeline B per-window Sharpe ratios
  - `eq_zscore.jpg`                      — Equation (1), z-score standardisation
  - `eq_reward.jpg`                      — Equation (2), clipped reward signal
  - `eq_sharpe.jpg`                      — Equation (3), annualised Sharpe ratio
  - `eq_hypotheses.jpg`                  — Equation (4), paired-test hypotheses

The notebook contains no global side effects beyond writing those files; each
cell can be re-run in isolation.


## Setup


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
from matplotlib.patches import FancyBboxPatch, Ellipse, Rectangle
from matplotlib.ticker import FuncFormatter

matplotlib.rcParams['font.family'] = 'serif'

# Figures go to ./figures/ next to this notebook
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

# Data paths (relative to notebook location)
PRICES_GZ      = Path("..") / "sp500_it_processed_full.csv.gz"
RESULTS_DIR    = Path("results")
PIPE_A_RET_CSV = RESULTS_DIR / "pipeline_a_test_daily_returns.csv"
PIPE_B_RES_CSV = RESULTS_DIR / "pipeline_b_test_results.csv"

print(f"Saving figures to: {FIG_DIR.resolve()}")


## Figure 1 — Methodological pipeline


In [ ]:
# ---------- palette (matching reference style) ----------
DATA_COLOR  = "#F5B82E"   # orange  — data
PREP_COLOR  = "#C6E0B4"   # green   — data preparation
MODEL_COLOR = "#9DC3E6"   # blue    — modeling
EVAL_COLOR  = "#F8CBAD"   # peach   — result evaluation
EDGE        = "#000000"
ARROW       = "#000000"
FONT        = "serif"

fig, ax = plt.subplots(figsize=(14, 9), dpi=200)
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.axis("off")


def rect(x, y, w, h, color, text, fontsize=12.5, bold=False):
    ax.add_patch(Rectangle((x, y), w, h, facecolor=color,
                           edgecolor=EDGE, linewidth=1.1))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center",
            family=FONT, fontsize=fontsize,
            fontweight="bold" if bold else "normal", linespacing=1.4)


def cylinder(x, y, w, h, color, text, fontsize=12):
    eh = h * 0.20
    ax.add_patch(Rectangle((x, y + eh / 2), w, h - eh,
                           facecolor=color, edgecolor=EDGE, linewidth=1.1))
    ax.add_patch(Ellipse((x + w / 2, y + h - eh / 2), w, eh,
                         facecolor=color, edgecolor=EDGE, linewidth=1.1))
    ax.add_patch(Ellipse((x + w / 2, y + eh / 2), w, eh,
                         facecolor=color, edgecolor=EDGE, linewidth=1.1))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center",
            family=FONT, fontsize=fontsize, linespacing=1.4)


def arrow(x1, y1, x2, y2):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="-|>", color=ARROW, lw=1.3,
                                mutation_scale=18))


def line(xs, ys):
    ax.plot(xs, ys, color=ARROW, lw=1.3, solid_capstyle="round")


# ═══ DATA (orange cylinders) ════════════════════════════════════════════════
cylinder(2, 80, 20, 14, DATA_COLOR,
         "Stock data\nS&P 500 IT (47 stocks)\nCRSP daily, 1999–2024", fontsize=11.5)
cylinder(2, 62, 20, 14, DATA_COLOR,
         "Macro data\nCBOE VIX\n(FRED)", fontsize=11.5)

# merge bus into feature engineering
line([22, 25], [87, 87])
line([22, 25], [69, 69])
line([25, 25], [87, 69])
arrow(25, 78, 26, 78)

# ═══ DATA PREPARATION (green, top row) ══════════════════════════════════════
rect(26, 71, 17, 14, PREP_COLOR, "Feature\nengineering\n(14 per stock)")
rect(46, 71, 16, 14, PREP_COLOR, "Train / validation /\ntest split")
rect(65, 71, 15, 14, PREP_COLOR, "z-score\nstandardization")
rect(82, 71, 16, 14, PREP_COLOR, "State\nconstruction\n(705-D)")

arrow(43, 78, 46, 78)
arrow(62, 78, 65, 78)
arrow(80, 78, 82, 78)

# ═══ MODELING (blue, centred) ═══════════════════════════════════════════════
rect(37, 54, 26, 9, MODEL_COLOR, "Modeling\n(Stable-Baselines3)", bold=True)
# state -> modeling (down, across to centre, down)
line([90, 90], [71, 67])
line([50, 90], [67, 67])
arrow(50, 67, 50, 63)

rect(30, 38, 19, 11, MODEL_COLOR, "PPO\ncontinuous weights\n(47-dim softmax)")
rect(51, 38, 19, 11, MODEL_COLOR, "DQN\ndiscrete actions\n(9 templates)")

# branch modeling -> PPO / DQN
line([50, 50], [54, 51])
line([39.5, 60.5], [51, 51])
arrow(39.5, 51, 39.5, 49)
arrow(60.5, 51, 60.5, 49)

# ═══ PIPELINES (white callout, in the open right-hand space) ════════════════
ax.add_patch(Rectangle((71, 39), 27, 18, facecolor="#FFFFFF",
                       edgecolor=EDGE, linewidth=1.1))
ax.text(84.5, 53.5, "Pipelines", ha="center", va="center",
        family=FONT, fontsize=13, fontweight="bold")
ax.text(84.5, 45.5,
        "A — fixed split\n(Train / Val / Test)\n\n"
        "B — walk-forward (6 windows)",
        ha="center", va="center", family=FONT, fontsize=11, linespacing=1.5)

# ═══ RESULTS (peach, centred) ═══════════════════════════════════════════════
rect(42, 26, 16, 8, EVAL_COLOR, "Results", bold=True)
# merge PPO / DQN -> results
line([39.5, 39.5], [38, 35.5])
line([60.5, 60.5], [38, 35.5])
line([39.5, 60.5], [35.5, 35.5])
arrow(50, 35.5, 50, 34)

# ═══ EVALUATION (peach, spread across bottom) ═══════════════════════════════
rect(16, 4, 28, 16, EVAL_COLOR,
     "Model evaluation\n\nSharpe · return\nvolatility · drawdown · turnover")
rect(56, 4, 30, 16, EVAL_COLOR,
     "Benchmarking\n\npaired t-test vs.\nEqual-Weight, Buy-Hold,\nMarket-Cap baselines")

# branch results -> evaluations
line([50, 50], [26, 23])
line([30, 71], [23, 23])
arrow(30, 23, 30, 20)
arrow(71, 23, 71, 20)

# ═══ LEGEND (left, in the open space) ═══════════════════════════════════════
legend_patches = [
    mpatches.Patch(facecolor=DATA_COLOR,  edgecolor=EDGE, label="Data"),
    mpatches.Patch(facecolor=PREP_COLOR,  edgecolor=EDGE, label="Data preparation"),
    mpatches.Patch(facecolor=MODEL_COLOR, edgecolor=EDGE, label="Modeling"),
    mpatches.Patch(facecolor=EVAL_COLOR,  edgecolor=EDGE, label="Result evaluation"),
]
ax.legend(handles=legend_patches, loc="upper left",
          bbox_to_anchor=(0.015, 0.55), fontsize=13, frameon=True,
          handlelength=1.5, handleheight=1.3, labelspacing=0.8,
          borderpad=1.0, prop={"family": FONT, "size": 13})

out_dir = FIG_DIR
plt.savefig(os.path.join(out_dir, "figure1_pipeline.png"),
            bbox_inches="tight", dpi=300, facecolor="white")
plt.close()
print("Saved figures/figure1_pipeline.png")


## Figures 2–4 — Z-score standardisation examples (AAPL)

Three illustrative side-by-side plots showing how train-window z-score
standardisation transforms a representative price series, a bounded momentum
oscillator (RSI), and a heavy-tailed volume series. All scaler parameters
(μ, σ) are estimated on the training window only (≤ 2018-12-31).


In [ ]:
# ── Load AAPL ──────────────────────────────────────────────────────────────
df = pd.read_csv(
    PRICES_GZ,
    usecols=['TICKER', 'date', 'PRC', 'ADJ_PRC', 'log_vol']
)
df = df[df['TICKER'] == 'AAPL'].copy()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Raw 14-day RSI (Wilder smoothing) from split-adjusted close.
# (The dataset's stored RSI column is already standardised, so we recompute
#  the raw 0–100 series here to illustrate the standardisation step.)
_delta = df['ADJ_PRC'].astype(float).diff()
_gain  = _delta.clip(lower=0)
_loss  = -_delta.clip(upper=0)
_avg_gain = _gain.ewm(alpha=1 / 14, min_periods=14, adjust=False).mean()
_avg_loss = _loss.ewm(alpha=1 / 14, min_periods=14, adjust=False).mean()
df['RSI'] = 100 - 100 / (1 + _avg_gain / _avg_loss)

# ── Boundaries ─────────────────────────────────────────────────────────────
train_end = pd.Timestamp('2018-12-31')
val_end   = pd.Timestamp('2021-12-31')

def zscore(series, train_mask):
    mu    = series[train_mask].mean()
    sigma = series[train_mask].std()
    return (series - mu) / sigma

train_mask = df['date'] <= train_end
df['PRC_z']     = zscore(df['PRC'],     train_mask)
df['RSI_z']     = zscore(df['RSI'],     train_mask)
df['log_vol_z'] = zscore(df['log_vol'], train_mask)

# ── Shared style ───────────────────────────────────────────────────────────
GREY = '#444444'
BLUE = '#2166AC'

TITLE_FS = 15
LABEL_FS = 13
TICK_FS  = 12
LEG_FS   = 11.5
BND_FS   = 11.5

out_dir = FIG_DIR


def fmt_ax(ax, ylabel=''):
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(5))
    ax.tick_params(labelsize=TICK_FS)
    ax.set_ylabel(ylabel, fontsize=LABEL_FS, family='serif')
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
    ax.grid(True, linewidth=0.4, alpha=0.5)


def add_boundaries(ax):
    t = ax.get_xaxis_transform()
    ax.axvline(train_end, color='#E31A1C', linestyle='--', linewidth=1.2, alpha=0.8)
    ax.axvline(val_end,   color='#FF7F00', linestyle='--', linewidth=1.2, alpha=0.8)
    ax.text(train_end, 0.5, ' train | val ', transform=t, fontsize=BND_FS,
            family='serif', color='#E31A1C', ha='right', va='center',
            rotation=90)
    ax.text(val_end, 0.5, ' val | test ', transform=t, fontsize=BND_FS,
            family='serif', color='#CC6600', ha='left', va='center',
            rotation=90)


def make_figure(raw_col, z_col, raw_ylabel, raw_title, z_title, fname,
                rsi_bands=False):
    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 4.2), dpi=300)
    fig.subplots_adjust(wspace=0.28)

    # raw panel
    ax0.plot(df['date'], df[raw_col], color=GREY, linewidth=0.9)
    if rsi_bands:
        ax0.axhline(70, color=BLUE, linewidth=0.8, linestyle='--', alpha=0.6,
                    label='Overbought (70)')
        ax0.axhline(30, color=BLUE, linewidth=0.8, linestyle='--', alpha=0.6,
                    label='Oversold (30)')
        ax0.legend(fontsize=LEG_FS, loc='lower left', framealpha=0.6)
    ax0.set_title(raw_title, fontsize=TITLE_FS, family='serif')
    fmt_ax(ax0, raw_ylabel)
    add_boundaries(ax0)

    # standardised panel
    ax1.plot(df['date'], df[z_col], color=GREY, linewidth=0.9)
    ax1.axhline(0, color='black', linewidth=0.6, linestyle=':')
    ax1.set_title(z_title, fontsize=TITLE_FS, family='serif')
    fmt_ax(ax1, 'Standard deviations')
    add_boundaries(ax1)

    plt.savefig(os.path.join(out_dir, fname),
                bbox_inches='tight', dpi=300, facecolor='white')
    plt.close()
    print(f'Saved figures/{fname}')


# ── Figure 2 — Closing price (PRC) → fig02_feat_prc ─────────────────────────
make_figure('PRC', 'PRC_z', 'USD',
            'Raw closing price — Apple Inc. (AAPL)',
            'z-score standardised closing price',
            'figure2_zscore_prc.png')

# ── Figure 3 (thesis) — Log volume → fig03_feat_vol ─────────────────────────
make_figure('log_vol', 'log_vol_z', 'log(1 + volume)',
            'Raw log volume — Apple Inc. (AAPL)',
            'z-score standardised log volume',
            'figure4_zscore_logvol.png')

# ── Figure 4 (thesis) — RSI → fig04_feat_rsi ────────────────────────────────
make_figure('RSI', 'RSI_z', 'RSI (0–100)',
            'Raw RSI — Apple Inc. (AAPL)',
            'z-score standardised RSI',
            'figure3_zscore_rsi.png', rsi_bands=True)


## Figure 5 — PPO actor network architecture


In [ ]:
"""
Figure 5 — PPO Actor network architecture (NN-SVG style, monochrome).
Input state -> two hidden layers (64, ReLU) -> softmax -> portfolio weights.
Saved to figures/figure5_ppo_architecture.png
"""

NODE_FC   = "#FFFFFF"
NODE_EC   = "#333333"
EDGE_COL  = "#9AA0A6"
TEXT_COL  = "#1A1A1A"
SUB_COL   = "#666666"
FONT      = "serif"

R       = 0.26          # node radius
Y_HI, Y_LO = 6.5, 1.0   # vertical extent of every column
HEAD_Y  = 7.6
SUB_Y   = 0.30

fig, ax = plt.subplots(figsize=(15.0, 6.0), dpi=300)
ax.set_xlim(0, 15.0)
ax.set_ylim(0, 8.0)
ax.set_aspect("equal")
ax.axis("off")


def nodes(cx, ys):
    for y in ys:
        ax.add_patch(plt.Circle((cx, y), R, facecolor=NODE_FC,
                                 edgecolor=NODE_EC, linewidth=1.6, zorder=3))


def connect(x0, ys0, x1, ys1):
    for y0 in ys0:
        for y1 in ys1:
            ax.plot([x0 + R, x1 - R], [y0, y1], color=EDGE_COL,
                    linewidth=0.5, alpha=0.55, zorder=1)


def vdots(cx, y):
    for dy in (-0.22, 0.0, 0.22):
        ax.plot(cx, y + dy, marker="o", color=NODE_EC, markersize=2.4, zorder=4)


def header(cx, lines):
    ax.text(cx, HEAD_Y, "\n".join(lines), ha="center", va="top",
            fontsize=13.5, family=FONT, fontweight="bold",
            color=TEXT_COL, linespacing=1.25)


def sublabel(cx, text):
    ax.text(cx, SUB_Y, text, ha="center", va="bottom", fontsize=10.5,
            family=FONT, color=SUB_COL, style="italic")


def side_label(x, y, text, side):
    """Bold underlined label to the left or right of a node."""
    ha = "right" if side == "left" else "left"
    ax.text(x, y, text, ha=ha, va="center", fontsize=13,
            family=FONT, fontweight="bold", color=TEXT_COL)
    w = len(text) * 0.105
    x_end = x - w if side == "left" else x + w
    ax.plot([x, x_end], [y - 0.30, y - 0.30], color=TEXT_COL, linewidth=1.2)


# ── column x positions ──────────────────────────────────────────────────────
X_IN, X_H1, X_H2, X_OUT = 2.6, 5.8, 9.0, 12.2

# ── node y positions ────────────────────────────────────────────────────────
ys_in  = [6.5, 5.1, 3.7, 1.0]                 # F1, F2, F3, F705 (+ dots)
ys_h1  = list(np.linspace(Y_HI, Y_LO, 8))
ys_h2  = list(np.linspace(Y_HI, Y_LO, 8))
ys_out = [6.5, 5.1, 3.7, 1.0]                 # w1, w2, w3, w47 (+ dots)

# ── connections (draw first, under nodes) ───────────────────────────────────
connect(X_IN, ys_in, X_H1, ys_h1)
connect(X_H1, ys_h1, X_H2, ys_h2)
connect(X_H2, ys_h2, X_OUT, ys_out)

# ── nodes ───────────────────────────────────────────────────────────────────
nodes(X_IN, ys_in);  vdots(X_IN, 2.35)
nodes(X_H1, ys_h1)
nodes(X_H2, ys_h2)
nodes(X_OUT, ys_out); vdots(X_OUT, 2.35)

# ── headers + sublabels ─────────────────────────────────────────────────────
header(X_IN,  ["State", "input"]);       sublabel(X_IN,  "705-D vector")
header(X_H1,  ["Hidden", "layer 1"]);    sublabel(X_H1,  "64 units, ReLU")
header(X_H2,  ["Hidden", "layer 2"]);    sublabel(X_H2,  "64 units, ReLU")
header(X_OUT, ["Portfolio", "weights"]); sublabel(X_OUT, "softmax → weights")

# ── input feature labels ────────────────────────────────────────────────────
for y, lbl in zip(ys_in, ["Feature 1", "Feature 2", "Feature 3", "Feature 705"]):
    side_label(X_IN - R - 0.18, y, lbl, "left")

# ── output weight labels ────────────────────────────────────────────────────
for y, lbl in zip(ys_out, ["Weight Stock 1", "Weight Stock 2",
                           "Weight Stock 3", "Weight Stock 47"]):
    side_label(X_OUT + R + 0.18, y, lbl, "right")

out_dir = FIG_DIR
plt.savefig(os.path.join(out_dir, "figure5_ppo_architecture.png"),
            bbox_inches="tight", dpi=300, facecolor="white")
plt.close()
print("Saved figures/figure5_ppo_architecture.png")


## Figure 6 — DQN Q-network architecture


In [ ]:
"""
Figure 6 — DQN Q-network architecture (NN-SVG style, monochrome).
Input state -> two hidden layers (64, ReLU) -> nine Q-values (one per template).
The agent selects the template with the highest Q-value (argmax).
Saved to figures/figure6_dqn_architecture.png
"""

NODE_FC   = "#FFFFFF"
NODE_EC   = "#333333"
EDGE_COL  = "#9AA0A6"
TEXT_COL  = "#1A1A1A"
SUB_COL   = "#666666"
FONT      = "serif"

R       = 0.26          # node radius
Y_HI, Y_LO = 6.5, 1.0   # vertical extent of every column
HEAD_Y  = 7.6
SUB_Y   = 0.30

fig, ax = plt.subplots(figsize=(15.0, 6.0), dpi=300)
ax.set_xlim(0, 15.0)
ax.set_ylim(0, 8.0)
ax.set_aspect("equal")
ax.axis("off")


def nodes(cx, ys):
    for y in ys:
        ax.add_patch(plt.Circle((cx, y), R, facecolor=NODE_FC,
                                 edgecolor=NODE_EC, linewidth=1.6, zorder=3))


def connect(x0, ys0, x1, ys1):
    for y0 in ys0:
        for y1 in ys1:
            ax.plot([x0 + R, x1 - R], [y0, y1], color=EDGE_COL,
                    linewidth=0.5, alpha=0.55, zorder=1)


def vdots(cx, y):
    for dy in (-0.22, 0.0, 0.22):
        ax.plot(cx, y + dy, marker="o", color=NODE_EC, markersize=2.4, zorder=4)


def header(cx, lines):
    ax.text(cx, HEAD_Y, "\n".join(lines), ha="center", va="top",
            fontsize=13.5, family=FONT, fontweight="bold",
            color=TEXT_COL, linespacing=1.25)


def sublabel(cx, text):
    ax.text(cx, SUB_Y, text, ha="center", va="bottom", fontsize=10.5,
            family=FONT, color=SUB_COL, style="italic")


def side_label(x, y, text, side):
    """Bold underlined label to the left or right of a node."""
    ha = "right" if side == "left" else "left"
    ax.text(x, y, text, ha=ha, va="center", fontsize=13,
            family=FONT, fontweight="bold", color=TEXT_COL)
    w = len(text) * 0.105
    x_end = x - w if side == "left" else x + w
    ax.plot([x, x_end], [y - 0.30, y - 0.30], color=TEXT_COL, linewidth=1.2)


# ── column x positions ──────────────────────────────────────────────────────
X_IN, X_H1, X_H2, X_OUT = 2.6, 5.8, 9.0, 12.2

# ── node y positions ────────────────────────────────────────────────────────
ys_in  = [6.5, 5.1, 3.7, 1.0]                 # F1, F2, F3, F705 (+ dots)
ys_h1  = list(np.linspace(Y_HI, Y_LO, 8))
ys_h2  = list(np.linspace(Y_HI, Y_LO, 8))
ys_out = list(np.linspace(Y_HI, Y_LO, 9))     # nine Q-values, one per template

# ── connections (draw first, under nodes) ───────────────────────────────────
connect(X_IN, ys_in, X_H1, ys_h1)
connect(X_H1, ys_h1, X_H2, ys_h2)
connect(X_H2, ys_h2, X_OUT, ys_out)

# ── nodes ───────────────────────────────────────────────────────────────────
nodes(X_IN, ys_in);  vdots(X_IN, 2.35)
nodes(X_H1, ys_h1)
nodes(X_H2, ys_h2)
nodes(X_OUT, ys_out)

# ── headers + sublabels ─────────────────────────────────────────────────────
header(X_IN,  ["State", "input"]);       sublabel(X_IN,  "705-D vector")
header(X_H1,  ["Hidden", "layer 1"]);    sublabel(X_H1,  "64 units, ReLU")
header(X_H2,  ["Hidden", "layer 2"]);    sublabel(X_H2,  "64 units, ReLU")
header(X_OUT, ["Q-values"]);             sublabel(X_OUT, "argmax → action")

# ── input feature labels ────────────────────────────────────────────────────
for y, lbl in zip(ys_in, ["Feature 1", "Feature 2", "Feature 3", "Feature 705"]):
    side_label(X_IN - R - 0.18, y, lbl, "left")

# ── output Q-value labels (one per portfolio template) ──────────────────────
q_labels = [f"Q(s, a{i})" for i in range(1, 10)]
for y, lbl in zip(ys_out, q_labels):
    side_label(X_OUT + R + 0.18, y, lbl, "right")

out_dir = FIG_DIR
plt.savefig(os.path.join(out_dir, "figure6_dqn_architecture.png"),
            bbox_inches="tight", dpi=300, facecolor="white")
plt.close()
print("Saved figures/figure6_dqn_architecture.png")


## Figure 7 — Pipeline A fixed train/validation/test split


In [ ]:
"""
Figure 7 — Pipeline A: fixed train / validation / test split timeline.
Saved to figures/figure7_pipeline_a.png
"""

COLOUR = {'train': '#4472C4', 'val': '#ED7D31', 'test': '#FFC000'}

TRAIN = (1999, 2018)
VAL   = (2019, 2021)
TEST  = (2022, 2024)

fig, ax = plt.subplots(figsize=(12, 3.0), dpi=300)
ax.set_xlim(1998.5, 2025.5)
ax.set_ylim(0, 1.6)
ax.axis('off')


def draw_block(ax, x0, x1, colour):
    width = x1 - x0 + 1
    rect = plt.Rectangle((x0 - 0.5, 0.35), width, 0.55,
                         facecolor=colour, edgecolor='white', linewidth=0.8)
    ax.add_patch(rect)


draw_block(ax, *TRAIN, COLOUR['train'])
draw_block(ax, *VAL,   COLOUR['val'])
draw_block(ax, *TEST,  COLOUR['test'])

# Training label — wide block, label inside
ax.text((TRAIN[0] + TRAIN[1]) / 2, 0.625,
        f'Training\n{TRAIN[0]}–{TRAIN[1]}',
        ha='center', va='center', fontsize=13,
        family='serif', color='white', fontweight='bold', linespacing=1.4)

# Validation label — narrow block, label above with leader line
mid_val = (VAL[0] + VAL[1]) / 2
ax.annotate(f'Validation\n{VAL[0]}–{VAL[1]}',
            xy=(mid_val, 0.90), xytext=(mid_val - 1.6, 1.35),
            fontsize=12, family='serif', color=COLOUR['val'],
            fontweight='bold', ha='center', va='bottom', linespacing=1.3,
            arrowprops=dict(arrowstyle='-', color=COLOUR['val'],
                            lw=0.9, shrinkA=0, shrinkB=2))

# Test label — narrow block, label above with leader line
mid_test = (TEST[0] + TEST[1]) / 2
ax.annotate(f'Test\n{TEST[0]}–{TEST[1]}',
            xy=(mid_test, 0.90), xytext=(mid_test + 1.6, 1.35),
            fontsize=12, family='serif', color='#B8860B',
            fontweight='bold', ha='center', va='bottom', linespacing=1.3,
            arrowprops=dict(arrowstyle='-', color='#B8860B',
                            lw=0.9, shrinkA=0, shrinkB=2))

# Year ticks
for y in range(1999, 2025, 3):
    ax.text(y, 0.25, str(y), ha='center', va='top', fontsize=11, family='serif',
            color='#555555')
    ax.plot([y, y], [0.31, 0.35], color='#999999', linewidth=0.6)

out_dir = FIG_DIR
plt.savefig(os.path.join(out_dir, 'figure7_pipeline_a.png'),
            bbox_inches='tight', dpi=300, facecolor='white')
plt.close()
print('Saved figures/figure7_pipeline_a.png')


## Figure 8 — Pipeline B walk-forward windows


In [ ]:
"""
Figure 8 — Pipeline B: walk-forward windows.
Years run along the x-axis; each row is one walk-forward window.
Saved to figures/figure8_pipeline_b.png
"""

COLOUR = {'train': '#4472C4', 'val': '#ED7D31', 'test': '#FFC000'}
EDGE   = '#FFFFFF'

YEARS = list(range(1999, 2024))   # 1999–2023

WINDOWS = {
    1: {'train': range(1999, 2014), 'val': range(2014, 2018), 'test': [2018]},
    2: {'train': range(2000, 2015), 'val': range(2015, 2019), 'test': [2019]},
    3: {'train': range(2001, 2016), 'val': range(2016, 2020), 'test': [2020]},
    4: {'train': range(2002, 2017), 'val': range(2017, 2021), 'test': [2021]},
    5: {'train': range(2003, 2018), 'val': range(2018, 2022), 'test': [2022]},
    6: {'train': range(2004, 2019), 'val': range(2019, 2023), 'test': [2023]},
}


def role(wd, year):
    if year in wd['train']: return 'train'
    if year in wd['val']:   return 'val'
    if year in wd['test']:  return 'test'
    return None


n_cols = len(YEARS)   # years on x-axis
n_rows = 6            # windows on y-axis

fig, ax = plt.subplots(figsize=(13, 4.2), dpi=300)
ax.set_xlim(0, n_cols)
ax.set_ylim(0, n_rows)
ax.set_aspect('equal')
ax.axis('off')

# ── Cells (years across, windows down) ──────────────────────────────────────
for w in range(1, 7):
    y = n_rows - w          # window 1 at the top row
    for col, year in enumerate(YEARS):
        r = role(WINDOWS[w], year)
        fc = COLOUR[r] if r else '#F2F2F2'
        rect = plt.Rectangle((col, y), 1, 1,
                             facecolor=fc, edgecolor=EDGE, linewidth=0.5)
        ax.add_patch(rect)

# ── Year labels (x-axis) ────────────────────────────────────────────────────
for col, year in enumerate(YEARS):
    ax.text(col + 0.5, -0.2, str(year), ha='center', va='top',
            rotation=90, fontsize=10, family='serif', color='#444444')

# ── Window labels (y-axis) ──────────────────────────────────────────────────
for w in range(1, 7):
    y = n_rows - w + 0.5
    ax.text(-0.25, y, f'Window {w}', ha='right', va='center',
            fontsize=11.5, family='serif', fontweight='bold')

# ── Legend ──────────────────────────────────────────────────────────────────
patches = [
    mpatches.Patch(facecolor=COLOUR['train'], edgecolor='grey', linewidth=0.4, label='Training set'),
    mpatches.Patch(facecolor=COLOUR['val'],   edgecolor='grey', linewidth=0.4, label='Validation set'),
    mpatches.Patch(facecolor=COLOUR['test'],  edgecolor='grey', linewidth=0.4, label='Test set'),
]
ax.legend(handles=patches, loc='upper center',
          bbox_to_anchor=(0.5, -0.18), ncol=3,
          fontsize=11, framealpha=0.8,
          handlelength=1.4, handleheight=1.2,
          borderpad=0.7, columnspacing=1.6)

out_dir = FIG_DIR
plt.savefig(os.path.join(out_dir, 'figure8_pipeline_b.png'),
            bbox_inches='tight', dpi=300, facecolor='white')
plt.close()
print('Saved figures/figure8_pipeline_b.png')


## Figure 9 — Pipeline A test-set equity curves (2022–2024)

Top panel: cumulative portfolio value ($1 invested) for all five strategies.
Bottom panel: drawdown of the two RL agents (mean across 3 random seeds, with
a ±1 σ shaded band).


In [ ]:
"""
Pipeline A — test-set equity curves (2022–2024).

Top panel : cumulative portfolio value ($1 invested).
Bottom panel : drawdown for the two RL agents only.

PPO and DQN are shown as the mean across three random seeds,
with a ±1 σ shaded band.

Output: figure_test_a_equity.jpg (300 dpi, JPG)
"""


matplotlib.rcParams['font.family'] = 'serif'

# ── palette ──────────────────────────────────────────────────────────
C_BH   = '#808080'   # grey   — Buy-and-Hold
C_EW   = '#4472C4'   # blue   — Equal-Weight
C_MC   = '#70AD47'   # green  — Mkt-Cap Weight
C_PPO  = '#C00000'   # dark red
C_DQN  = '#ED7D31'   # orange

# ── paths ─────────────────────────────────────────────────────────────
# ── load daily returns ────────────────────────────────────────────────
df = pd.read_csv(PIPE_A_RET_CSV)
df = df.dropna().reset_index(drop=True)
n  = len(df)

# ── trading dates for test window ────────────────────────────────────
raw_dates = pd.read_csv(PRICES_GZ, usecols=['date'])
raw_dates['date'] = pd.to_datetime(raw_dates['date'])
test_dates = (raw_dates['date']
              .drop_duplicates()
              .sort_values()
              .loc[lambda s: (s >= '2022-01-01') & (s <= '2024-12-31')]
              .reset_index(drop=True))
dates = test_dates.iloc[:n].values          # align length with returns

# ── cumulative return helper ──────────────────────────────────────────
def cumret(s):
    return (1 + s).cumprod().values

# baselines
cum_bh = cumret(df['Buy_Hold'])
cum_ew = cumret(df['Equal_Weight'])
cum_mc = cumret(df['MktCap_Weight'])

# agents — stack seeds then average
ppo_seeds = ['PPO_s42', 'PPO_s123', 'PPO_s456']
dqn_seeds = ['DQN_s42', 'DQN_s123', 'DQN_s456']

ppo_cum = np.stack([cumret(df[s]) for s in ppo_seeds])
dqn_cum = np.stack([cumret(df[s]) for s in dqn_seeds])

ppo_mean, ppo_std = ppo_cum.mean(0), ppo_cum.std(0)
dqn_mean, dqn_std = dqn_cum.mean(0), dqn_cum.std(0)

# ── drawdown helper ───────────────────────────────────────────────────
def drawdown(arr):
    peak = np.maximum.accumulate(arr)
    return (arr - peak) / peak

ppo_dd = np.stack([drawdown(cumret(df[s])) for s in ppo_seeds]).mean(0)
dqn_dd = np.stack([drawdown(cumret(df[s])) for s in dqn_seeds]).mean(0)

# ── figure ────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(10, 6.2),
    gridspec_kw={'height_ratios': [3, 1]},
    sharex=True
)
fig.subplots_adjust(hspace=0.06)

# — top panel: cumulative value —
ax1.plot(dates, cum_bh, color=C_BH,  lw=0.9, ls='--', label='Buy-and-Hold',  zorder=3)
ax1.plot(dates, cum_ew, color=C_EW,  lw=0.9, ls='--', label='Equal-Weight',  zorder=3)
ax1.plot(dates, cum_mc, color=C_MC,  lw=0.9, ls='--', label='Mkt-Cap Weight', zorder=3)

ax1.fill_between(dates, ppo_mean - ppo_std, ppo_mean + ppo_std,
                 color=C_PPO, alpha=0.12, zorder=2)
ax1.plot(dates, ppo_mean, color=C_PPO, lw=1.5,
         label='PPO (mean ±1σ, 3 seeds)', zorder=4)

ax1.fill_between(dates, dqn_mean - dqn_std, dqn_mean + dqn_std,
                 color=C_DQN, alpha=0.12, zorder=2)
ax1.plot(dates, dqn_mean, color=C_DQN, lw=1.5,
         label='DQN (mean ±1σ, 3 seeds)', zorder=4)

ax1.axhline(1.0, color='#555555', lw=0.55, ls=':', zorder=1)
ax1.set_ylabel('Cumulative portfolio value ($1 invested)',
               fontsize=12.5, family='serif', labelpad=6)
ax1.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'${y:.2f}'))
ax1.legend(loc='upper left', fontsize=11, framealpha=0.92,
           edgecolor='#cccccc', prop={'family': 'serif', 'size': 11})
ax1.grid(axis='y', ls=':', lw=0.5, color='#cccccc')
ax1.tick_params(labelsize=11)
for sp in ['top', 'right']:
    ax1.spines[sp].set_visible(False)


# — bottom panel: drawdowns —
ax2.fill_between(dates, ppo_dd, 0, color=C_PPO, alpha=0.30, label='PPO drawdown')
ax2.fill_between(dates, dqn_dd, 0, color=C_DQN, alpha=0.30, label='DQN drawdown')
ax2.plot(dates, ppo_dd, color=C_PPO, lw=0.85)
ax2.plot(dates, dqn_dd, color=C_DQN, lw=0.85)
ax2.axhline(0, color='#555555', lw=0.55, ls=':')

# cap y-axis so no positive ticks appear above the zero line
dd_floor = min(ppo_dd.min(), dqn_dd.min())
ax2.set_ylim(dd_floor * 1.08, 0.01)   # small positive headroom

ax2.set_ylabel('Drawdown (RL Agents)', fontsize=12.5, family='serif', labelpad=6)
ax2.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'{y:.0%}'))
ax2.legend(loc='lower left', fontsize=11, framealpha=0.92,
           edgecolor='#cccccc', prop={'family': 'serif', 'size': 11})
ax2.grid(axis='y', ls=':', lw=0.5, color='#cccccc')
ax2.tick_params(labelsize=11)
for sp in ['top', 'right']:
    ax2.spines[sp].set_visible(False)

# x-axis: quarterly ticks
ax2.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=0, ha='center', fontsize=11)

plt.tight_layout()
out_dir = FIG_DIR
out_path = os.path.join(out_dir, 'figure9_test_a_equity.png')
plt.savefig(out_path, bbox_inches='tight', dpi=300, facecolor='white')
plt.close()
print('Saved figures/figure9_test_a_equity.png')


## Figure 10 — Pipeline B per-window Sharpe ratios (2018–2023)


In [ ]:
"""
Pipeline B — per-window Sharpe ratio grouped bar chart (2018–2023).

Five strategies shown as grouped bars for each test year.
Bear years (2018, 2022) and bull years (2019, 2023) are immediately
visible from the bar heights.

Output: figure8_test_b_sharpe_bar.jpg (300 dpi, JPG)
"""


matplotlib.rcParams['font.family'] = 'serif'

# ── palette ───────────────────────────────────────────────────────────
C_BH  = '#808080'
C_EW  = '#4472C4'
C_MC  = '#70AD47'
C_PPO = '#C00000'
C_DQN = '#ED7D31'

# ── paths ─────────────────────────────────────────────────────────────
# ── load data ─────────────────────────────────────────────────────────
df = pd.read_csv(PIPE_B_RES_CSV)

# window → test year mapping
YEAR_MAP = {1: 2018, 2: 2019, 3: 2020, 4: 2021, 5: 2022, 6: 2023}
df['year'] = df['window'].map(YEAR_MAP)

# pivot to (year × strategy) table
strategies = [
    ('Buy_Hold',    C_BH,  'Buy-and-Hold'),
    ('Equal_Weight',C_EW,  'Equal-Weight'),
    ('MktCap_Weight',C_MC, 'Mkt-Cap Weight'),
    ('PPO',         C_PPO, 'PPO'),
    ('DQN',         C_DQN, 'DQN'),
]
years = [2018, 2019, 2020, 2021, 2022, 2023]
n_strategies = len(strategies)
n_years      = len(years)

# ── bar geometry ──────────────────────────────────────────────────────
group_width = 0.75          # total width of one year's group of bars
bar_width   = group_width / n_strategies
x = np.arange(n_years)

# offsets so bars are centred around each year tick
offsets = np.linspace(-(group_width - bar_width) / 2,
                       (group_width - bar_width) / 2,
                       n_strategies)

# ── figure ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4.6), dpi=300)

for i, (col, color, label) in enumerate(strategies):
    heights = [
        df.loc[(df['algorithm'] == col) & (df['year'] == yr), 'sharpe'].values[0]
        for yr in years
    ]
    ax.bar(x + offsets[i], heights, width=bar_width,
           color=color, alpha=0.88, zorder=3, label=label)

# zero line
ax.axhline(0, color='#333333', lw=0.8, zorder=4)

# light vertical separators between year groups
for xi in x[:-1]:
    ax.axvline(xi + 0.5, color='#dddddd', lw=0.7, ls='--', zorder=2)

ax.set_xticks(x)
ax.set_xticklabels([str(y) for y in years], fontsize=12, family='serif')
ax.set_ylabel('Sharpe Ratio (annualised)', fontsize=12.5, family='serif', labelpad=6)
ax.tick_params(axis='y', labelsize=11)
ax.grid(axis='y', ls=':', lw=0.5, color='#cccccc', zorder=1)
for sp in ['top', 'right']:
    ax.spines[sp].set_visible(False)

# legend
legend_handles = [
    mpatches.Patch(color=color, alpha=0.88, label=label)
    for _, color, label in strategies
]
ax.legend(handles=legend_handles,
          loc='lower center', bbox_to_anchor=(0.5, 1.01),
          ncol=5, frameon=False,
          columnspacing=1.4, handletextpad=0.5,
          prop={'family': 'serif', 'size': 11})

# headroom so the tallest bars never reach the legend row
ymin, ymax = ax.get_ylim()
ax.set_ylim(ymin, ymax * 1.08)

plt.tight_layout()
out_dir = FIG_DIR
out_path = os.path.join(out_dir, 'figure10_test_b_sharpe_bar.png')
plt.savefig(out_path, bbox_inches='tight', dpi=300, facecolor='white')
plt.close()
print('Saved figures/figure10_test_b_sharpe_bar.png')


## Equation renders

Static mathtext renders of the four equations cited in Chapter 4. Numbered to
match the body of the thesis (1: z-score, 2: clipped reward, 3: annualised
Sharpe, 4: paired-test hypotheses).


In [ ]:
# Equation (1) — z-score standardisation
fig, ax = plt.subplots(figsize=(9, 1.4), dpi=300)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.text(0.5, 0.5,
        r"$\tilde{x}_{i,t} \;=\; \dfrac{x_{i,t} \,-\, \mu_{i}^{\,\mathrm{train}}}{\sigma_{i}^{\,\mathrm{train}}}$",
        ha="center", va="center", fontsize=22, family="serif")
ax.text(0.98, 0.5, "(1)", ha="right", va="center", fontsize=14, family="serif")
out = FIG_DIR / "eq_zscore.jpg"
plt.savefig(out, bbox_inches="tight", dpi=300, facecolor="white", pad_inches=0.2)
plt.show(); print(f"Saved {out}")


In [ ]:
# Equation (2) — clipped reward signal
fig, ax = plt.subplots(figsize=(9, 2.2), dpi=300)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.text(0.5, 0.68,
        r"$r_t \;=\; \mathrm{clip}\!\left(100\cdot r^{\,p}_{t+1},\;-10,\;10\right)$",
        ha="center", va="center", fontsize=22, family="serif")
ax.text(0.5, 0.28,
        r"$r^{\,p}_{t+1} \;=\; \mathbf{w}_t^{\top}\mathbf{R}_{t+1} \;-\; c\,\|\Delta\mathbf{w}\|_1$",
        ha="center", va="center", fontsize=18, family="serif", color="#444444")
ax.text(0.98, 0.5, "(2)", ha="right", va="center", fontsize=14, family="serif")
out = FIG_DIR / "eq_reward.jpg"
plt.savefig(out, bbox_inches="tight", dpi=300, facecolor="white", pad_inches=0.2)
plt.show(); print(f"Saved {out}")


In [ ]:
# Equation (3) — annualised Sharpe ratio
fig, ax = plt.subplots(figsize=(9, 1.4), dpi=300)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.text(0.5, 0.5,
        r"$\mathrm{Sharpe}_{\,\mathrm{annualised}} \;=\; \dfrac{\bar{r}}{\sigma_{r}}\,\cdot\,\sqrt{252}$",
        ha="center", va="center", fontsize=22, family="serif")
ax.text(0.98, 0.5, "(3)", ha="right", va="center", fontsize=14, family="serif")
out = FIG_DIR / "eq_sharpe.jpg"
plt.savefig(out, bbox_inches="tight", dpi=300, facecolor="white", pad_inches=0.2)
plt.show(); print(f"Saved {out}")


In [ ]:
# Equation (4) — paired-test hypotheses
fig, ax = plt.subplots(figsize=(9, 2.0), dpi=300)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.text(0.5, 0.68,
        r"$H_0\colon\; \mathrm{Sharpe}_{\mathrm{agent}} \;\leq\; \mathrm{Sharpe}_{\mathrm{EW}}$",
        ha="center", va="center", fontsize=20, family="serif")
ax.text(0.5, 0.28,
        r"$H_1\colon\; \mathrm{Sharpe}_{\mathrm{agent}} \;>\; \mathrm{Sharpe}_{\mathrm{EW}}$",
        ha="center", va="center", fontsize=20, family="serif")
ax.text(0.98, 0.5, "(4)", ha="right", va="center", fontsize=14, family="serif")
out = FIG_DIR / "eq_hypotheses.jpg"
plt.savefig(out, bbox_inches="tight", dpi=300, facecolor="white", pad_inches=0.2)
plt.show(); print(f"Saved {out}")
